In [1]:
import pandas as pd
import numpy as np

In [2]:
data = pd.read_csv("autos_clean.csv", delimiter=',' ,encoding='latin-1')

In [3]:

data.head()

,price,vehicleType,gearbox,powerPS,kilometer,fuelType,brand,notRepairedDamage,model_final,listingAge,car_age
0,9800,suv,automatik,163,125000,diesel,jeep,NaN,grand cherokee,22.533171,22
1,1500,kleinwagen,manuell,75,150000,benzin,volkswagen,nein,golf,0.736308,25
2,3600,kleinwagen,manuell,69,90000,diesel,skoda,nein,fabia,6.428715,18
3,650,limousine,manuell,102,150000,benzin,bmw,ja,316i,2.803553,31
4,2200,cabrio,manuell,109,150000,benzin,peugeot,nein,206,4.762951,22


In [4]:
data = data.drop(columns=["listingAge", "notRepairedDamage", "car_age"])

In [5]:
#Splitting the data

from sklearn.model_selection import train_test_split

numeric_features = ['powerPS', 'kilometer']
categorical_features = ['vehicleType', 'gearbox', 'model_final', 'fuelType', 'brand', ]
#data[categorical_features] = data[categorical_features].fillna("unknown")



y = data['price'].copy()
X = data[categorical_features + numeric_features].copy()

X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, random_state = 42, test_size = 0.2)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, random_state = 42, test_size = 0.2)





In [6]:
#Standard Random Forest Regressor
import time
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

ohe = OneHotEncoder(handle_unknown = "ignore", sparse_output = False, max_categories = 50)

seed = 42

def make_forest(criterion = "squared_error", max_depth = 3, min_samples_leaf = 5):
    prepare = ColumnTransformer([
        ("numeric", SimpleImputer(strategy = "mean"), numeric_features),
        ("categorical", ohe, categorical_features)
    ])
    return Pipeline([
        ("prepare", prepare),
        ("forest", RandomForestRegressor(criterion=criterion, max_depth=max_depth, min_samples_leaf=min_samples_leaf, n_jobs=-1, random_state = seed)),
         ])

#Defining the hyperparameters and running the model:

def holdout_table(parameter_choices):
    rows, models = [], []
    for i, settings in enumerate(ParameterGrid(parameter_choices), start=1):
        t0 = time.time()
        model = make_forest(**settings).fit(X_train, y_train)
        models.append(model)
        probs_train = model.predict(X_train)
        probs_val = model.predict(X_val)
        
        rows.append({
            "settings":settings,
            "training MAE": mean_absolute_error(y_train, probs_train), #which measure to use?!
            "validation MAE": mean_absolute_error(y_val, probs_val), #which measure to use?!
            "training RMSE": root_mean_squared_error(y_train, probs_train),
            "validation RMSE": root_mean_squared_error(y_val, probs_val),
            "val_r2": r2_score(y_val, probs_val),
            "avg_leaves": sum(t.get_n_leaves() for t in model.named_steps["forest"].estimators_) / len(model.named_steps["forest"].estimators_),
        })
        print(f"[{i}/9] Finished {settings} in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), models

depth_results, depth_models = holdout_table({
    "criterion" : ["squared_error"], "max_depth" : [3, 6, 9], "min_samples_leaf" : [5, 10, 20]
})
depth_labels = [str(settings["max_depth"]) for settings in depth_results["settings"]]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(depth_results.round(3))




[1/9] Finished {'criterion': 'squared_error', 'max_depth': 3, 'min_samples_leaf': 5} in 4.5s
[2/9] Finished {'criterion': 'squared_error', 'max_depth': 3, 'min_samples_leaf': 10} in 4.9s
[3/9] Finished {'criterion': 'squared_error', 'max_depth': 3, 'min_samples_leaf': 20} in 4.7s
[4/9] Finished {'criterion': 'squared_error', 'max_depth': 6, 'min_samples_leaf': 5} in 7.2s
[5/9] Finished {'criterion': 'squared_error', 'max_depth': 6, 'min_samples_leaf': 10} in 7.6s
[6/9] Finished {'criterion': 'squared_error', 'max_depth': 6, 'min_samples_leaf': 20} in 7.2s
[7/9] Finished {'criterion': 'squared_error', 'max_depth': 9, 'min_samples_leaf': 5} in 8.3s
[8/9] Finished {'criterion': 'squared_error', 'max_depth': 9, 'min_samples_leaf': 10} in 11.9s
[9/9] Finished {'criterion': 'squared_error', 'max_depth': 9, 'min_samples_leaf': 20} in 11.2s


,settings,training MAE,validation MAE,training RMSE,validation RMSE,val_r2,avg_leaves
0,"{'criterion': 'squared_error', 'max_depth': 3,...",3259.508,3271.009,5044.598,5101.734,0.573,8.00
1,"{'criterion': 'squared_error', 'max_depth': 3,...",3259.508,3271.009,5044.598,5101.734,0.573,8.00
2,"{'criterion': 'squared_error', 'max_depth': 3,...",3259.508,3271.009,5044.598,5101.734,0.573,8.00
3,"{'criterion': 'squared_error', 'max_depth': 6,...",2511.959,2520.789,3982.620,4067.242,0.729,64.00
4,"{'criterion': 'squared_error', 'max_depth': 6,...",2511.048,2519.396,3983.260,4065.667,0.729,63.99
5,"{'criterion': 'squared_error', 'max_depth': 6,...",2510.819,2523.023,3999.705,4086.266,0.726,63.94
6,"{'criterion': 'squared_error', 'max_depth': 9,...",2163.515,2191.169,3421.897,3557.221,0.792,433.96
7,"{'criterion': 'squared_error', 'max_depth': 9,...",2171.676,2196.433,3464.616,3586.103,0.789,402.09
8,"{'criterion': 'squared_error', 'max_depth': 9,...",2190.255,2208.316,3533.337,3647.581,0.782,371.50


In [7]:
#Standard HistGradientBoostRegressor
import time
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, root_mean_squared_error
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import ParameterGrid

ohe = OrdinalEncoder(handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=-2,
    max_categories=254,)
cat_indices = list(range(len(numeric_features), len(numeric_features) + len(categorical_features)))

seed = 42

def make_forest(loss = "absolute_error", learning_rate = 0.1, max_iter= 300, max_leaf_nodes = 31, min_samples_leaf = 5, l2_regularization=0.0,):
    prepare = ColumnTransformer([
        ("numeric", "passthrough", numeric_features),
        ("categorical", ohe, categorical_features),
    ])
    pipe = Pipeline([
        ("prepare", prepare),
        ("forest", HistGradientBoostingRegressor(loss=loss,
            learning_rate=learning_rate,
            max_iter=max_iter,
            max_leaf_nodes=max_leaf_nodes,
            min_samples_leaf=min_samples_leaf,
            l2_regularization=l2_regularization,
            categorical_features=cat_indices,
            early_stopping=True,      
            validation_fraction=0.1,
            n_iter_no_change=15,
            random_state=seed,)),
         ])
    return TransformedTargetRegressor(
        regressor=pipe,
        func=np.log1p,
        inverse_func=np.expm1,)
        
#Defining the hyperparameters and running the model:

def holdout_table(parameter_choices):
    rows, models = [], []
    for i, settings in enumerate(ParameterGrid(parameter_choices), start=1):
        t0 = time.time()
        model = make_forest(**settings).fit(X_train, y_train)
        models.append(model)
        probs_train = model.predict(X_train)
        probs_val = model.predict(X_val)

        predictors = model.regressor_.named_steps["forest"]._predictors
        
        rows.append({
            "settings":settings,
            "training MAE": mean_absolute_error(y_train, probs_train), #which measure to use?!
            "validation MAE": mean_absolute_error(y_val, probs_val), #which measure to use?!
            "training RMSE": root_mean_squared_error(y_train, probs_train),
            "validation RMSE": root_mean_squared_error(y_val, probs_val),
            "val_r2": r2_score(y_val, probs_val),
            "avg_nodes": sum(p[0].nodes.shape[0] for p in predictors) / len(predictors),
        })
        print(f"[{i}/9] Finished {settings} in {time.time() - t0:.1f}s")
    return pd.DataFrame(rows), models

depth_results, depth_models = holdout_table({
    "loss" : ["absolute_error"], "learning_rate": [0.05], "max_iter" : [1000], "max_leaf_nodes" : [127, 255], "min_samples_leaf" : [15, 20, 30], "l2_regularization": [0.1],
})
leaf_labels = [str(settings["max_leaf_nodes"]) for settings in depth_results["settings"]]

with pd.option_context("display.max_rows", None, "display.max_columns", None):
    display(depth_results.round(3))




[1/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 127, 'min_samples_leaf': 15} in 76.3s
[2/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 127, 'min_samples_leaf': 20} in 69.9s
[3/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 127, 'min_samples_leaf': 30} in 62.6s
[4/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 255, 'min_samples_leaf': 15} in 118.2s
[5/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 255, 'min_samples_leaf': 20} in 115.8s
[6/9] Finished {'l2_regularization': 0.1, 'learning_rate': 0.05, 'loss': 'absolute_error', 'max_iter': 1000, 'max_leaf_nodes': 255, 'min_samples_leaf': 30} in 107.9s


,settings,training MAE,validation MAE,training RMSE,validation RMSE,val_r2,avg_nodes
0,"{'l2_regularization': 0.1, 'learning_rate': 0....",1441.485,1580.064,2651.765,2855.410,0.866,253.0
1,"{'l2_regularization': 0.1, 'learning_rate': 0....",1445.380,1582.077,2671.915,2874.368,0.864,253.0
2,"{'l2_regularization': 0.1, 'learning_rate': 0....",1452.428,1582.999,2691.238,2871.425,0.865,253.0
3,"{'l2_regularization': 0.1, 'learning_rate': 0....",1358.656,1552.007,2538.478,2824.906,0.869,509.0
4,"{'l2_regularization': 0.1, 'learning_rate': 0....",1357.594,1548.191,2546.752,2821.585,0.869,509.0
5,"{'l2_regularization': 0.1, 'learning_rate': 0....",1383.279,1554.730,2605.062,2837.131,0.868,509.0
